# Notebook 11 — Escalado de features 📏

En el **NB10** creaste features nuevas. Hoy te toca un tema técnico pero **crítico**: poner todas tus features en la **misma escala** para que ningún modelo las trate injustamente.

## El problema en una imagen

```
    Sin escalar:                 Con escalar (StandardScaler):
    age   ∈ [0, 80]              age   ≈ media 0,  std 1
    fare  ∈ [0, 512]             fare  ≈ media 0,  std 1
    pclass ∈ [1, 3]              pclass ≈ media 0, std 1

    ⟹ fare "pesa" más           ⟹ todas pesan igual
       en distancias y           el modelo decide
       en gradientes             qué importa
```

## ¿Por qué importa?

- **KNN, k-means**: usan distancias euclidianas; `fare` (max 512) dominaría a `age` (max 80) y la distancia se calcularía casi sólo sobre `fare`.
- **Redes neuronales, regresión regularizada (Ridge/Lasso)**: trabajan con gradientes; features grandes "tiran" más del modelo.
- **Árboles y Random Forest**: **no** necesitan escalado (deciden splits por columna).

## Objetivos de aprendizaje

1. Entender **por qué** algunos modelos necesitan que las features estén en la misma escala.
2. Aplicar `StandardScaler` — media 0, desviación estándar 1.
3. Aplicar `MinMaxScaler` — rango [0, 1].
4. Entender la **regla de oro**: `fit` solo con `X_train`, `transform` con `X_train` y `X_test`.
5. Visualizar el efecto del escalado con boxplots.

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Same cleanup as NB06 — go back to seaborn's titanic for continuity
df = sns.load_dataset("titanic")
df_clean = df.drop(columns=["deck"]).copy()
df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())
df_clean = df_clean.dropna(subset=["embarked"]).reset_index(drop=True)

# Encode categoricals as in NB09
df_encoded = pd.get_dummies(
    df_clean,
    columns=["sex", "embarked"],
    drop_first=True,
    dtype=int,
)

features = ["age", "pclass", "sibsp", "parch", "fare",
            "sex_male", "embarked_Q", "embarked_S"]
X = df_encoded[features]
y = df_encoded["survived"]  # we'll switch to classification next notebook — this is just for setup

print(f"X: {X.shape}, y: {y.shape}")
X.describe().round(2)

---

## 2. Mira las escalas — el problema concreto

Fíjate en los rangos de `age`, `fare` y `pclass`. Si un modelo basado en distancias calcula:

```
    distancia entre dos pasajeros = √((Δage)² + (Δpclass)² + (Δfare)² + …)
```

`fare` (que puede diferir en 500 unidades entre dos pasajeros) **domina por completo** sobre `pclass` (que difiere como máximo en 2). El modelo se vuelve ciego a las features chicas.

In [ ]:
# Visualize the scale problem with a boxplot of all numeric features
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([X[col] for col in features], tick_labels=features)
ax.set_title("Escalas crudas de las features — fare domina")
ax.set_ylabel("Valor")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---

## 3. Train/test split — primero divides, luego escalas

Esto es **crítico**: el escalador se debe **entrenar (`fit`) sólo con `X_train`**. Si lo entrenas con todo (`X_train + X_test`), estás dejando que **información del test "fugue" al train** — eso es **data leakage** y arruina la honestidad de tu evaluación.

```
    ✅ Forma correcta:                ❌ Forma incorrecta:
    ─────────────────                ────────────────────
    scaler.fit(X_train)              scaler.fit(X)  # con todo
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
```

Así de simple: **fittea con train, transforma con ambos**.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

---

## 4. `StandardScaler` — media 0, desviación 1

`StandardScaler` resta la media y divide por la desviación estándar **de cada columna**:

```
    x_scaled = (x - mean) / std
```

Después de escalar, cada feature tiene:

- **Media** ≈ 0
- **Desviación estándar** ≈ 1

Es la opción **por defecto** para la mayoría de modelos de scikit-learn.

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)                              # learn mean and std from TRAIN
X_train_scaled = scaler.transform(X_train)       # apply to train
X_test_scaled = scaler.transform(X_test)         # apply same transform to test
```

> 💡 Atajo: `scaler.fit_transform(X_train)` hace `fit` + `transform` en una sola llamada. **Nunca** uses `fit_transform` sobre `X_test`.

### 🏋️ Ejercicio 1 — Aplicar `StandardScaler`

1. Importa `StandardScaler` desde `sklearn.preprocessing`.
2. Crea **`scaler`** = `StandardScaler()`.
3. Fittea **únicamente** sobre `X_train` (usa `fit_transform` para train).
4. Guarda el resultado en **`X_train_scaled`** (un `np.ndarray`).
5. Transforma `X_test` y guarda el resultado en **`X_test_scaled`**.

In [ ]:
from sklearn.preprocessing import StandardScaler

# YOUR CODE HERE
scaler = None
X_train_scaled = None
X_test_scaled = None

In [ ]:
# Tests — verify the exercise was completed correctly
assert isinstance(scaler, StandardScaler), \
    f"scaler must be a StandardScaler, got {type(scaler).__name__}"

assert hasattr(scaler, "mean_"), "scaler is not fitted — did you call fit / fit_transform?"

# Means and std should have been learned from TRAIN only
assert scaler.mean_.shape == (8,), f"scaler.mean_ shape must be (8,), got {scaler.mean_.shape}"
expected_train_mean = X_train.mean().values
assert np.allclose(scaler.mean_, expected_train_mean, rtol=1e-5), \
    "scaler.mean_ must match X_train.mean() — did you accidentally fit on X or X_test?"

# Output shapes
assert isinstance(X_train_scaled, np.ndarray), \
    f"X_train_scaled must be a numpy ndarray, got {type(X_train_scaled).__name__}"
assert X_train_scaled.shape == X_train.shape, \
    f"X_train_scaled shape must be {X_train.shape}, got {X_train_scaled.shape}"
assert X_test_scaled.shape == X_test.shape, \
    f"X_test_scaled shape must be {X_test.shape}, got {X_test_scaled.shape}"

# After scaling, X_train_scaled mean ≈ 0 and std ≈ 1 column-wise
col_means = X_train_scaled.mean(axis=0)
col_stds = X_train_scaled.std(axis=0)
assert np.allclose(col_means, 0, atol=1e-7), \
    f"X_train_scaled columns must have mean ≈ 0, got {col_means}"
assert np.allclose(col_stds, 1, atol=1e-7), \
    f"X_train_scaled columns must have std ≈ 1, got {col_stds}"

# But X_test_scaled does NOT need to have mean=0/std=1 — that would mean you re-fit on test
print("✅ ¡Todos los tests pasaron! StandardScaler aplicado correctamente.")
print(f"\nX_train_scaled — media por columna: {col_means.round(2)}")
print(f"X_train_scaled — std por columna:   {col_stds.round(2)}")
print(f"\nX_test_scaled  — media por columna: {X_test_scaled.mean(axis=0).round(2)}  ← no es exactamente 0")
print("   (eso es correcto: test no se usó para fittear)")

---

## 5. `MinMaxScaler` — todo entre 0 y 1

Otra opción común es **`MinMaxScaler`**, que mapea linealmente cada columna al rango **[0, 1]**:

```
    x_scaled = (x - min) / (max - min)
```

| Cuándo elegir `MinMaxScaler` vs `StandardScaler` |
|---|
| `StandardScaler` es el **default** para datos con distribución más o menos normal |
| `MinMaxScaler` se prefiere cuando quieres un rango **acotado** (p. ej. imágenes con píxeles 0-255 → 0-1) |
| Ambos sufren con **outliers**; para datos con muchos outliers existe `RobustScaler` |

### 🏋️ Ejercicio 2 — Aplicar `MinMaxScaler`

1. Importa `MinMaxScaler` desde `sklearn.preprocessing`.
2. Crea **`mm_scaler`** = `MinMaxScaler()`.
3. Fittea sobre `X_train` y guarda el resultado en **`X_train_mm`**.
4. Transforma `X_test` y guarda en **`X_test_mm`**.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# YOUR CODE HERE
mm_scaler = None
X_train_mm = None
X_test_mm = None

In [ ]:
# Tests — verify the exercise was completed correctly
assert isinstance(mm_scaler, MinMaxScaler), \
    f"mm_scaler must be a MinMaxScaler, got {type(mm_scaler).__name__}"
assert hasattr(mm_scaler, "data_min_"), "mm_scaler is not fitted"

# After fit on train, every column of X_train_mm should be in [0, 1]
assert isinstance(X_train_mm, np.ndarray), "X_train_mm must be a numpy ndarray"
assert X_train_mm.shape == X_train.shape
assert X_train_mm.min() >= 0.0 - 1e-9, f"X_train_mm min must be >= 0, got {X_train_mm.min()}"
assert X_train_mm.max() <= 1.0 + 1e-9, f"X_train_mm max must be <= 1, got {X_train_mm.max()}"

# Each column should reach both 0 and 1 (since we fit on this exact data)
col_mins = X_train_mm.min(axis=0)
col_maxs = X_train_mm.max(axis=0)
assert np.allclose(col_mins, 0, atol=1e-9), \
    f"Each column of X_train_mm should reach 0, got mins {col_mins}"
assert np.allclose(col_maxs, 1, atol=1e-9), \
    f"Each column of X_train_mm should reach 1, got maxs {col_maxs}"

# X_test_mm can occasionally have values < 0 or > 1 (an unseen extreme value)
print("✅ ¡Tests pasaron! MinMaxScaler aplicado correctamente.")
print(f"\nX_train_mm — rango por columna: min={col_mins.round(2)}")
print(f"                                  max={col_maxs.round(2)}")

---

## 6. Visualizar el efecto del escalado

Antes el `fare` aplastaba al resto. Después, todas las features viven en rangos comparables.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].boxplot([X_train[col] for col in features], tick_labels=features)
axes[0].set_title("Sin escalar")
axes[0].set_ylabel("Valor")
axes[0].tick_params(axis="x", rotation=45)

axes[1].boxplot([X_train_scaled[:, i] for i in range(X_train_scaled.shape[1])],
                tick_labels=features)
axes[1].set_title("StandardScaler  (media 0, std 1)")
axes[1].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[1].tick_params(axis="x", rotation=45)

axes[2].boxplot([X_train_mm[:, i] for i in range(X_train_mm.shape[1])],
                tick_labels=features)
axes[2].set_title("MinMaxScaler  (rango [0, 1])")
axes[2].set_ylim(-0.1, 1.1)
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

---

## 7. ¿Cuándo escalar — y cuándo no?

| Modelo | ¿Necesita escalado? |
|---|---|
| **KNN, K-Means** | ✅ Sí — usa distancias |
| **Regresión logística** | ✅ Recomendado, sobre todo con regularización |
| **Redes neuronales** | ✅ Sí — los gradientes lo agradecen |
| **Ridge / Lasso (regresión regularizada)** | ✅ Sí — la penalización depende del tamaño de los coeficientes |
| **Árbol de decisión** | ❌ No — los splits son por umbral en una sola columna |
| **Random Forest, Gradient Boosting** | ❌ No — son árboles |
| **Regresión lineal simple (sin regularización)** | 🟡 No es estrictamente necesario, pero hace que los coeficientes sean comparables |

> 💡 **Regla práctica**: ante la duda, escala. No hace daño y es requisito para varios modelos.

---

## 8. Resumen — ¿qué aprendiste?

Las features en escalas muy distintas hacen que ciertos modelos se vuelvan **ciegos** a las variables más pequeñas. Escalar las pone en igualdad de condiciones.

### Conceptos clave

| Concepto | Idea |
|---|---|
| **`StandardScaler`** | `(x - mean) / std` — media 0, desviación 1 |
| **`MinMaxScaler`** | `(x - min) / (max - min)` — rango [0, 1] |
| **Regla de oro** | `fit` sólo con `X_train`; `transform` ambos |
| **Data leakage** | Fittear con todos los datos (incluido el test) "contamina" la evaluación |

### Reglas prácticas

1. **Siempre divide en train/test primero**, luego escala. Nunca al revés.
2. **`fit` sólo con `X_train`**, `transform` con ambos. Esa simetría es lo que mantiene la evaluación honesta.
3. **Árboles no requieren** escalado — si tu pipeline final solo usa árboles, te puedes ahorrar el paso.
4. Después de escalar, los **coeficientes de la regresión lineal** se vuelven comparables — el más grande en valor absoluto es el más influyente.

### Lo que viene en el NB12

Hasta ahora has predicho un número (`fare`). En el **NB12** das el gran salto a **clasificación**: predecir una categoría (¿sobrevivió o no?) con `LogisticRegression`. Y vas a usar todas las features escaladas que acabas de aprender a preparar.